# Training-only controller feasibility — user-provided discussion inspiration

This is a cheap falsification check, not a pressure predictor. The user supplied `../kagglediscussion.md`; its source hash and the stated course-rule conflict are recorded in the protocol. Current V014 confirmation is unchanged. No external code or data is used.

Independently derived hypotheses: for a proportional controller with linearly varying desired pressure, `u(t)=a+b*t-Kp*P(t-lag)`. For a non-decaying PI controller, `delta_u=-Kp*delta_P-Ki*dt*P+A*dt+B*dt*t`. Test contemporaneous and one-step-lagged pressure alignments. These assumptions may be wrong: target shape, timing, clipping, perturbations and integral-state decay can all invalidate them.

Choose64 original-A-fit breaths per R/C combination, before reading pressure. For each, fit coefficients on the first70% of eligible inspired rows and inspect control residuals on the remaining rows of that same training breath. This diagnostic uses their known pressure throughout; it does **not** predict unseen pressure and must never be reported as validation MAE. Original A-validation and E-validation are not used. A near-exact-control-consistency gate was frozen before calculation. All raw values retain float64 CSV precision. No model or submission is promoted by this check.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, time
import numpy as np
import pandas as pd
root=Path.cwd()
out=root/"outputs/controller_feasibility_v015"
out.mkdir(parents=True,exist_ok=True)
started=time.monotonic()
planpath=root/"research/controller_feasibility_protocol_v015.json"
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert sha(planpath)=="4797698d79be753b1de3099f5623225d2f3b112d27b636e4664d5cf2ea6df555"
plan=json.loads(planpath.read_text())
assert sha(root/"data/train.csv")==plan["approved_train_sha256"]
assert sha(root/plan["original_A_split_file"])==plan["A_split_file_sha256"]
assert sha(Path(plan["source_document"]))==plan["source_document_sha256"]
# Metadata only: choose complete original-A-training breaths before parsing labels.
meta=pd.read_csv(root/"data/train.csv",usecols=["breath_id","R","C"],dtype={"breath_id":"int32","R":"int8","C":"int8"}).iloc[::80].reset_index(drop=True)
assert len(meta)==67905
with np.load(root/plan["original_A_split_file"],allow_pickle=False) as z:
    fit_ids=z["A_fit"].copy();validation_ids=z["A_validation"].copy()
fit_mask=meta.breath_id.isin(fit_ids)
rng=np.random.default_rng(plan["sample_seed"])
selected=[]
for _,g in meta[fit_mask].groupby(["R","C"],sort=True):
    selected.extend(rng.choice(g.index.to_numpy(),size=64,replace=False).tolist())
selected=np.array(sorted(selected),dtype=np.int64)
selected_ids=meta.breath_id.to_numpy()[selected]
assert len(selected)==576 and not np.intersect1d(selected_ids,validation_ids).size
selected_set=set(selected.tolist())
# skiprows excludes every other complete breath before the CSV parser reads pressure.
frame=pd.read_csv(root/"data/train.csv",usecols=["breath_id","R","C","time_step","u_in","u_out","pressure"],skiprows=lambda i:i>0 and (i-1)//80 not in selected_set,dtype={"breath_id":"int32","R":"int8","C":"int8","time_step":"float64","u_in":"float64","u_out":"int8","pressure":"float64"})
assert len(frame)==576*80 and np.array_equal(frame.breath_id.to_numpy()[::80],selected_ids)
assert frame.breath_id.isin(fit_ids).all() and not frame.breath_id.isin(validation_ids).any()
rows=[]
for bid,b in frame.groupby("breath_id",sort=False):
    t=b.time_step.to_numpy();u=b.u_in.to_numpy();pressure=b.pressure.to_numpy();phase=b.u_out.to_numpy()
    for equation in plan["equations"]:
        lag=1 if equation.endswith("lag1") else 0
        k=np.arange(2,80)
        ok=(phase[k]==0)&(phase[k-1]==0)&(phase[k-lag]==0)&(phase[k-lag-1]==0)&(u[k]>1e-6)&(u[k]<100-1e-6)
        if equation.startswith("PI"):
            ok&=(u[k-1]>1e-6)&(u[k-1]<100-1e-6)
        k=k[ok]
        if len(k)<12:continue
        if equation.startswith("PI"):
            dt=t[k]-t[k-1]
            X=np.column_stack([-(pressure[k-lag]-pressure[k-lag-1]),-dt*pressure[k-lag],dt,dt*t[k]])
            target=u[k]-u[k-1]
            kp_col=0
        else:
            X=np.column_stack([np.ones(len(k)),t[k],-pressure[k-lag]])
            target=u[k]
            kp_col=2
        cut=max(8,int(.7*len(k)))
        if len(k)-cut<3:continue
        scale=np.maximum(np.sqrt((X[:cut]**2).mean(0)),1e-12)
        Z=X/scale
        if np.linalg.matrix_rank(Z[:cut])<X.shape[1]:continue
        condition=float(np.linalg.cond(Z[:cut]))
        coef=np.linalg.lstsq(Z[:cut],target[:cut],rcond=None)[0]/scale
        prediction=X@coef
        residual=prediction-target
        rows.append(dict(breath_id=int(bid),R=int(b.R.iloc[0]),C=int(b.C.iloc[0]),equation=equation,rows=len(k),fit_rows=cut,diagnostic_holdout_rows=len(k)-cut,kp=float(coef[kp_col]),ki=float(coef[1]) if equation.startswith("PI") else None,condition=condition,fit_median_abs_control_residual=float(np.median(np.abs(residual[:cut]))),diagnostic_median_abs_control_residual=float(np.median(np.abs(residual[cut:]))),diagnostic_mean_abs_control_residual=float(np.abs(residual[cut:]).mean()),positive_kp=bool(coef[kp_col]>0),near_exact_control_consistency=bool(coef[kp_col]>0 and condition<1e10 and np.median(np.abs(residual[cut:]))<1e-4)))
records=pd.DataFrame(rows)
records.to_csv(out/"training_only_equation_fits.csv",index=False)
summary=[]
for name,g in records.groupby("equation",sort=True):
    summary.append(dict(equation=name,eligible_training_breaths=len(g),median_fit_control_residual=float(g.fit_median_abs_control_residual.median()),median_diagnostic_control_residual=float(g.diagnostic_median_abs_control_residual.median()),positive_kp_fraction=float(g.positive_kp.mean()),near_exact_count=int(g.near_exact_control_consistency.sum()),near_exact_fraction_of_sample=float(g.near_exact_control_consistency.sum()/576)))
report=dict(status="complete_training_only_diagnostic",protocol_sha256=sha(planpath),source_document_sha256=plan["source_document_sha256"],sample_training_breaths=576,selected_A_fit_breath_ids=selected_ids.tolist(),original_A_validation_excluded=True,E_validation_not_used=True,raw_float64_precision=True,diagnostic_units="valve control, not pressure MAE",equations=summary,gate_passed=bool(any(q["near_exact_fraction_of_sample"]>=.1 for q in summary)),interpretation="Checks conditional control-equation consistency using known pressure within original A training breaths; no unseen-pressure predictor, no recovery-coverage estimate, no new validation score",new_submission_created=False,runtime_seconds=time.monotonic()-started,completed_utc=datetime.now(timezone.utc).isoformat())
(out/"summary.json").write_text(json.dumps(report,indent=2)+"\n")
print(pd.DataFrame(summary).to_string(index=False))
print("Training-only feasibility gate:",report["gate_passed"])
print("Runtime seconds:",round(report["runtime_seconds"],2))
print("These residuals are in valve units and are not pressure-prediction MAEs.")


  equation  eligible_training_breaths  median_fit_control_residual  median_diagnostic_control_residual  positive_kp_fraction  near_exact_count  near_exact_fraction_of_sample
PI_current                        492                     0.037917                            0.185052              0.725610                13                       0.022569
   PI_lag1                        492                     0.335451                            0.942009              0.477642                 3                       0.005208
 P_current                        552                     0.464499                            1.455323              0.568841                18                       0.031250
    P_lag1                        552                     0.801345                            2.432545              0.420290                 3                       0.005208
Training-only feasibility gate: False
Runtime seconds: 2.2
These residuals are in valve units and are not pressure-prediction MAEs